# Phase 1: LIDC-IDRI Dataset Preprocessing & 3D Patch Extraction
## Project: A Hybrid Explainable AI Framework for Accurate Lung Nodule Detection and Malignancy Prediction

This notebook implements the complete **Phase 1 pipeline** for the LIDC-IDRI dataset:
1. **Environment Setup & Drive Integration**: Auto-installs dependencies and mounts Google Drive.
2. **DICOM Ingestion & 1mm Isotropic Resampling**: Utilizes SimpleITK for spatial coordinate mapping and uniform $(1.0, 1.0, 1.0)$ mm voxel resampling.
3. **CT Lung Windowing & Normalization**: Clips Hounsfield Units to $[-1000, 400]$ HU and scales to $[0, 1]$.
4. **Expert Annotation Parsing (pylidc)**: Groups radiologist annotations, computes consensus malignancy scores (1–5 scale), and extracts clinical semantic attributes.
5. **3D Patch Extraction (64x64x64)**: Extracts bounding cubes centered at nodule centroids with boundary padding.
6. **Interactive Visualizations**: 2D axial/coronal/sagittal orthogonal views with nodule bounding boxes and HU histograms.
7. **PyTorch Dataset & DataLoader**: Custom `LIDCNoduleDataset` ready for Phase 2 Deep Learning and Radiomics modeling.

### Step 1: Install Dependencies & Setup Environment

In [ ]:
# Install required medical imaging and deep learning libraries
!pip install -q pydicom SimpleITK pylidc torch h5py pandas numpy matplotlib tqdm

In [ ]:
import os
import sys
import glob
import json
import logging
from pathlib import Path
from typing import Tuple, List, Dict, Any, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import SimpleITK as sitk
import pydicom
import torch
from torch.utils.data import Dataset, DataLoader
import h5py
from tqdm.notebook import tqdm

# Monkey-patch NumPy aliases for pylidc compatibility on modern NumPy (>= 1.24 / 2.0)
if not hasattr(np, 'int'):
    np.int = int
if not hasattr(np, 'float'):
    np.float = float
if not hasattr(np, 'bool'):
    np.bool = bool
if not hasattr(np, 'object'):
    np.object = object

# Check if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Mounted Google Drive successfully!")
else:
    print("Running in local / non-Colab environment.")

### Step 2: Configuration & Path Management

In [ ]:
# Set dataset paths (Update paths according to your folder structure)
if IN_COLAB:
    RAW_DATA_DIR = Path("/content/drive/MyDrive/Lung_Nodule_Project/raw_data")
    OUTPUT_DIR = Path("/content/drive/MyDrive/Lung_Nodule_Project/processed_patches")
else:
    # Local fallback paths
    RAW_DATA_DIR = Path("./data/raw_data")
    OUTPUT_DIR = Path("./data/processed_patches")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PATCHES_DIR = OUTPUT_DIR / "patches"
PATCHES_DIR.mkdir(parents=True, exist_ok=True)

# Configure ~/.pylidcrc to enable pylidc scan queries
config_path = Path.home() / ".pylidcrc"
config_content = f"""[pylidc]
path = {RAW_DATA_DIR.resolve()}
warn = False
"""
config_path.write_text(config_content)
print(f"Configured ~/.pylidcrc with dataset path: {RAW_DATA_DIR}")

# Import pylidc after configuration
import pylidc as pl
print(f"pylidc successfully imported! Version: {pl.__version__ if hasattr(pl, '__version__') else 'active'}")

### Step 3: Core Preprocessing Pipeline Functions
- **SimpleITK Series Reader**: Reads DICOM series with correct slice sorting and metadata.
- **Isotropic Resampler**: Re-grids CT volume into uniform $(1.0, 1.0, 1.0)$ mm spacing.
- **Lung Windowing**: Clips HU to $[-1000, 400]$ and normalizes to $[0, 1]$.

In [ ]:
def find_dicom_series_dir(scan_folder: Path) -> Optional[Path]:
    """Find the directory with the primary CT series (largest number of slices)."""
    candidate_dirs = []
    for root, _, files in os.walk(scan_folder):
        dcm_count = sum(1 for f in files if f.lower().endswith(".dcm") or not "." in f)
        if dcm_count > 10:
            candidate_dirs.append((Path(root), dcm_count))
    if not candidate_dirs:
        return None
    candidate_dirs.sort(key=lambda x: x[1], reverse=True)
    return candidate_dirs[0][0]

def load_dicom_volume_sitk(dicom_dir: Path) -> sitk.Image:
    """Load DICOM series into a 3D SimpleITK image with spatial metadata."""
    reader = sitk.ImageSeriesReader()
    series_ids = reader.GetGDCMSeriesIDs(str(dicom_dir))
    if not series_ids:
        file_names = reader.GetGDCMSeriesFileNames(str(dicom_dir))
    else:
        file_names = reader.GetGDCMSeriesFileNames(str(dicom_dir), series_ids[0])
    
    if not file_names:
        raise FileNotFoundError(f"No DICOM files found in: {dicom_dir}")
        
    reader.SetFileNames(file_names)
    image = reader.Execute()
    return image

def resample_volume_isotropic(
    image: sitk.Image,
    target_spacing: Tuple[float, float, float] = (1.0, 1.0, 1.0),
    default_value: float = -1000.0
) -> sitk.Image:
    """Resample 3D SimpleITK image to isotropic (1.0, 1.0, 1.0) mm spacing."""
    orig_spacing = np.array(image.GetSpacing(), dtype=np.float64)
    orig_size = np.array(image.GetSize(), dtype=np.int64)
    target_spacing = np.array(target_spacing, dtype=np.float64)

    new_size = np.round(orig_size * orig_spacing / target_spacing).astype(np.int64)

    resample = sitk.ResampleImageFilter()
    resample.SetInterpolator(sitk.sitkLinear)
    resample.SetOutputSpacing(target_spacing.tolist())
    resample.SetSize(new_size.tolist())
    resample.SetOutputDirection(image.GetDirection())
    resample.SetOutputOrigin(image.GetOrigin())
    resample.SetDefaultPixelValue(default_value)
    resample.SetOutputPixelType(sitk.sitkFloat32)

    return resample.Execute(image)

def apply_lung_window_and_normalize(
    volume_np: np.ndarray,
    min_hu: float = -1000.0,
    max_hu: float = 400.0
) -> np.ndarray:
    """Clip CT intensities to lung window [-1000, 400] HU and normalize to [0, 1]."""
    clipped = np.clip(volume_np, min_hu, max_hu).astype(np.float32)
    normalized = (clipped - min_hu) / (max_hu - min_hu)
    return normalized

def extract_3d_patch(
    volume_np: np.ndarray,
    centroid_zyx: Tuple[int, int, int],
    patch_size: Tuple[int, int, int] = (64, 64, 64),
    pad_value: float = 0.0
) -> Tuple[np.ndarray, bool]:
    """Extract a 3D subvolume patch (64x64x64) with zero-padding at boundaries."""
    cz, cy, cx = centroid_zyx
    pd, ph, pw = patch_size
    vz, vy, vx = volume_np.shape

    half_d, half_h, half_w = pd // 2, ph // 2, pw // 2
    z_min_req, z_max_req = cz - half_d, cz + (pd - half_d)
    y_min_req, y_max_req = cy - half_h, cy + (ph - half_h)
    x_min_req, x_max_req = cx - half_w, cx + (pw - half_w)

    z_min_vol, z_max_vol = max(0, z_min_req), min(vz, z_max_req)
    y_min_vol, y_max_vol = max(0, y_min_req), min(vy, y_max_req)
    x_min_vol, x_max_vol = max(0, x_min_req), min(vx, x_max_req)

    z_min_patch = z_min_vol - z_min_req
    z_max_patch = z_min_patch + (z_max_vol - z_min_vol)
    y_min_patch = y_min_vol - y_min_req
    y_max_patch = y_min_patch + (y_max_vol - y_min_vol)
    x_min_patch = x_min_vol - x_min_req
    x_max_patch = x_min_patch + (x_max_vol - x_min_vol)

    patch = np.full(patch_size, fill_value=pad_value, dtype=np.float32)
    is_padded = (z_min_req < 0 or z_max_req > vz or y_min_req < 0 or y_max_req > vy or x_min_req < 0 or x_max_req > vx)

    if (z_max_vol > z_min_vol) and (y_max_vol > y_min_vol) and (x_max_vol > x_min_vol):
        patch[z_min_patch:z_max_patch, y_min_patch:y_max_patch, x_min_patch:x_max_patch] = \
            volume_np[z_min_vol:z_max_vol, y_min_vol:y_max_vol, x_min_vol:x_max_vol]

    return patch, is_padded

print("Core preprocessing functions defined successfully!")

### Step 4: Annotation Parsing & Consensus Malignancy
Extract nodule clusters from `pylidc`, compute the consensus malignancy score ($1.0 - 5.0$), and map world coordinates into resampled voxel coordinates.

In [ ]:
def parse_scan_nodules(scan: pl.Scan, resampled_sitk_img: sitk.Image, original_sitk_img: sitk.Image) -> List[Dict[str, Any]]:
    """Extract nodule clusters, consensus malignancy ratings, and resampled voxel coordinates."""
    nodules = []
    try:
        clusters = scan.cluster_annotations()
    except Exception as e:
        print(f"Annotation clustering error for {scan.patient_id}: {e}")
        return []

    for c_idx, cluster in enumerate(clusters):
        if not cluster:
            continue
        
        # 1. Consensus Malignancy Calculation
        malignancies = [ann.malignancy for ann in cluster if ann.malignancy is not None]
        if not malignancies:
            continue
        consensus_mal = float(np.mean(malignancies))
        
        # Classification label: 0 (Benign), 1 (Malignant), -1 (Indeterminate)
        if consensus_mal < 3.0:
            mal_class = 0
        elif consensus_mal > 3.0:
            mal_class = 1
        else:
            mal_class = -1

        # 2. Centroid Mapping (DICOM voxel -> World Physical -> Resampled Voxel)
        world_pts = []
        for ann in cluster:
            ann_vox = ann.centroid  # [y, x, z]
            sitk_idx = (float(ann_vox[1]), float(ann_vox[0]), float(ann_vox[2]))
            try:
                world_pt = original_sitk_img.TransformContinuousIndexToPhysicalPoint(sitk_idx)
                world_pts.append(world_pt)
            except Exception:
                pass

        if not world_pts:
            continue
            
        avg_world_pt = np.mean(world_pts, axis=0)
        res_cont_idx = resampled_sitk_img.TransformPhysicalPointToContinuousIndex(tuple(avg_world_pt))
        res_voxel_zyx = (
            int(np.round(res_cont_idx[2])),
            int(np.round(res_cont_idx[1])),
            int(np.round(res_cont_idx[0]))
        )

        # 3. Radiologist semantic features
        nodule_entry = {
            "nodule_id": f"{scan.patient_id}_nodule_{c_idx:03d}",
            "patient_id": scan.patient_id,
            "num_annotations": len(cluster),
            "consensus_malignancy": consensus_mal,
            "malignancy_class": mal_class,
            "subtlety": float(np.mean([ann.subtlety for ann in cluster if ann.subtlety])),
            "sphericity": float(np.mean([ann.sphericity for ann in cluster if ann.sphericity])),
            "margin": float(np.mean([ann.margin for ann in cluster if ann.margin])),
            "spiculation": float(np.mean([ann.spiculation for ann in cluster if ann.spiculation])),
            "texture": float(np.mean([ann.texture for ann in cluster if ann.texture])),
            "centroid_world_xyz": tuple(avg_world_pt),
            "centroid_voxel_zyx": res_voxel_zyx,
        }
        nodules.append(nodule_entry)

    return nodules

print("Annotation parser defined successfully!")

### Step 5: Interactive Visualizations (Orthogonal CT Views & 3D Patches)
Inspect CT slices and extracted 3D nodule patches across Axial, Coronal, and Sagittal planes.

In [ ]:
def plot_nodule_orthogonal_views(volume_np: np.ndarray, centroid_zyx: Tuple[int, int, int], patch_np: np.ndarray, title: str = "Nodule Visualization"):
    """Plot Axial, Coronal, and Sagittal orthogonal cross-sections with nodule patch."""
    cz, cy, cx = centroid_zyx
    pz, py, px = patch_np.shape[0] // 2, patch_np.shape[1] // 2, patch_np.shape[2] // 2

    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    fig.suptitle(title, fontsize=16, fontweight="bold")

    # Full CT volume cross-sections
    axes[0, 0].imshow(volume_np[cz, :, :], cmap="bone", origin="lower")
    axes[0, 0].plot(cx, cy, 'r+', markersize=12, markeredgewidth=2)
    axes[0, 0].set_title(f"Full Volume Axial (Z={cz})")

    axes[0, 1].imshow(volume_np[:, cy, :], cmap="bone", origin="lower")
    axes[0, 1].plot(cx, cz, 'r+', markersize=12, markeredgewidth=2)
    axes[0, 1].set_title(f"Full Volume Coronal (Y={cy})")

    axes[0, 2].imshow(volume_np[:, :, cx], cmap="bone", origin="lower")
    axes[0, 2].plot(cy, cz, 'r+', markersize=12, markeredgewidth=2)
    axes[0, 2].set_title(f"Full Volume Sagittal (X={cx})")

    # 64x64x64 Cropped Patch cross-sections
    axes[1, 0].imshow(patch_np[pz, :, :], cmap="bone", origin="lower")
    axes[1, 0].set_title("64x64 Nodule Patch (Axial Center)")

    axes[1, 1].imshow(patch_np[:, py, :], cmap="bone", origin="lower")
    axes[1, 1].set_title("64x64 Nodule Patch (Coronal Center)")

    axes[1, 2].imshow(patch_np[:, :, px], cmap="bone", origin="lower")
    axes[1, 2].set_title("64x64 Nodule Patch (Sagittal Center)")

    for ax in axes.flat:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

### Step 6: Process Single Patient / Demonstration Run

In [ ]:
# Example: Process the first available patient in raw_data_dir
all_patient_dirs = sorted([p for p in RAW_DATA_DIR.iterdir() if p.is_dir() and p.name.startswith("LIDC-IDRI-")])
print(f"Found {len(all_patient_dirs)} patient scan directories in {RAW_DATA_DIR}.")

if all_patient_dirs:
    sample_patient = all_patient_dirs[0].name
    print(f"Demonstration processing for: {sample_patient}")
    
    dicom_dir = find_dicom_series_dir(all_patient_dirs[0])
    if dicom_dir:
        # 1. Load DICOM
        orig_sitk = load_dicom_volume_sitk(dicom_dir)
        print(f"Original Volume Size: {orig_sitk.GetSize()}, Spacing: {orig_sitk.GetSpacing()}")

        # 2. Resample to 1mm isotropic
        res_sitk = resample_volume_isotropic(orig_sitk, target_spacing=(1.0, 1.0, 1.0))
        print(f"Resampled Volume Size: {res_sitk.GetSize()}, Spacing: {res_sitk.GetSpacing()}")

        # 3. Window & Normalize
        vol_np = sitk.GetArrayFromImage(res_sitk)
        vol_norm = apply_lung_window_and_normalize(vol_np)

        # 4. Parse annotations via pylidc
        scan = pl.query(pl.Scan).filter(pl.Scan.patient_id == sample_patient).first()
        if scan:
            nodules = parse_scan_nodules(scan, res_sitk, orig_sitk)
            print(f"Extracted {len(nodules)} nodule(s).")
            
            if nodules:
                sample_nodule = nodules[0]
                patch, is_pad = extract_3d_patch(vol_norm, sample_nodule["centroid_voxel_zyx"], patch_size=(64, 64, 64))
                
                # Visualize
                title = f"{sample_nodule['nodule_id']} | Consensus Malignancy: {sample_nodule['consensus_malignancy']:.2f}/5.0"
                plot_nodule_orthogonal_views(vol_norm, sample_nodule["centroid_voxel_zyx"], patch, title=title)
else:
    print("No LIDC-IDRI scans found in RAW_DATA_DIR. Ensure your dataset folder is properly mounted.")

### Step 7: Batch Dataset Preprocessing
Preprocess all patient scans, extract 3D patches ($64\times64\times64$), save `.pt` tensor files, and export `manifest.csv`.

In [ ]:
def preprocess_full_dataset(raw_dir: Path, out_dir: Path, max_scans: Optional[int] = None) -> pd.DataFrame:
    """Process all patient scans and export manifest.csv."""
    patient_dirs = sorted([p for p in raw_dir.iterdir() if p.is_dir() and p.name.startswith("LIDC-IDRI-")])
    if max_scans:
        patient_dirs = patient_dirs[:max_scans]

    all_records = []
    print(f"Starting batch preprocessing for {len(patient_dirs)} patients...")

    for p_dir in tqdm(patient_dirs, desc="Preprocessing Patients"):
        patient_id = p_dir.name
        try:
            dicom_dir = find_dicom_series_dir(p_dir)
            if not dicom_dir:
                continue

            orig_sitk = load_dicom_volume_sitk(dicom_dir)
            res_sitk = resample_volume_isotropic(orig_sitk, target_spacing=(1.0, 1.0, 1.0))
            vol_np = sitk.GetArrayFromImage(res_sitk)
            vol_norm = apply_lung_window_and_normalize(vol_np)

            scan = pl.query(pl.Scan).filter(pl.Scan.patient_id == patient_id).first()
            if not scan:
                continue

            nodules = parse_scan_nodules(scan, res_sitk, orig_sitk)
            for nodule in nodules:
                patch, is_pad = extract_3d_patch(vol_norm, nodule["centroid_voxel_zyx"], patch_size=(64, 64, 64))
                patch_path = out_dir / "patches" / f"{nodule['nodule_id']}.pt"
                
                # Save as PyTorch tensor (1, 64, 64, 64)
                tensor_4d = torch.from_numpy(patch).unsqueeze(0).to(torch.float32)
                torch.save({"tensor": tensor_4d, "metadata": nodule}, patch_path)
                
                nodule_record = {**nodule, "patch_path": str(patch_path), "is_padded": is_pad}
                all_records.append(nodule_record)

        except Exception as e:
            print(f"Error processing {patient_id}: {e}")

    manifest_df = pd.DataFrame(all_records)
    manifest_path = out_dir / "manifest.csv"
    manifest_df.to_csv(manifest_path, index=False)
    print(f"\nCompleted! Saved {len(all_records)} patches. Manifest: {manifest_path}")
    return manifest_df

# Run preprocessing (Set max_scans=None for all scans or specify a number like 10 for a test batch)
# manifest_df = preprocess_full_dataset(RAW_DATA_DIR, OUTPUT_DIR, max_scans=10)

### Step 8: PyTorch 3D Nodule Dataset (Ready for Phase 2 Modeling)
A clean, efficient `Dataset` class for training 3D CNNs (e.g. 3D ResNet, 3D DenseNet) and hybrid XAI models.

In [ ]:
class LIDCNoduleDataset(Dataset):
    """PyTorch Dataset for 3D LIDC-IDRI Nodule Patches."""
    def __init__(self, manifest_csv: str, exclude_indeterminate: bool = True, transform=None):
        self.df = pd.read_csv(manifest_csv)
        if exclude_indeterminate:
            # Filter out indeterminate nodules (malignancy == 3.0, label = -1)
            self.df = self.df[self.df["malignancy_class"] != -1].reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.df.iloc[idx]
        data = torch.load(row["patch_path"], weights_only=False)
        patch_tensor = data["tensor"]  # Shape: (1, 64, 64, 64)

        if self.transform:
            patch_tensor = self.transform(patch_tensor)
            
        label = torch.tensor(row["malignancy_class"], dtype=torch.long)
        malignancy_score = torch.tensor(row["consensus_malignancy"], dtype=torch.float32)
        
        # Clinical semantic attributes
        attributes = torch.tensor([
            row.get("subtlety", 0.0),
            row.get("sphericity", 0.0),
            row.get("margin", 0.0),
            row.get("spiculation", 0.0),
            row.get("texture", 0.0)
        ], dtype=torch.float32)

        return {
            "image": patch_tensor,
            "label": label,
            "malignancy_score": malignancy_score,
            "attributes": attributes,
            "nodule_id": row["nodule_id"],
            "patient_id": row["patient_id"],
        }

print("LIDCNoduleDataset class defined and ready for Phase 2 training!")